In [4]:
# Standard libraries
import pandas as pd
import numpy as np
import re
from itertools import chain, combinations
from tqdm import tqdm

# Display settings
from IPython.display import display
pd.options.display.max_columns = None

# Scientific computing and statistics
from scipy.spatial import cKDTree
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Visualization libraries
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from mpl_toolkits.mplot3d import Axes3D 

# Scikit-learn: Data processing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Scikit-learn: Evaluation metrics
from sklearn.metrics import mean_squared_error, adjusted_rand_score, accuracy_score, \
    confusion_matrix, classification_report, precision_score, recall_score, f1_score, \
    precision_recall_fscore_support

# Scikit-learn: Machine learning models
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.mixture import GaussianMixture
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample


# Scikit-learn: Dimensionality reduction and clustering
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score
from sklearn.metrics import calinski_harabasz_score
from sklearn.metrics import davies_bouldin_score

# Scikit-learn: Regression models
from sklearn.linear_model import Lasso, LinearRegression, Ridge

# Scikit-learn: Model selection
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
target = 'is_workload_inj'
features = ['age', 'days_since_last_start', 'pitches_last_start', 'number_start', 
            'FB_usage', 'FB_velo', 'FB_spin', 
            'BB_usage', 'BB_velo', 'BB_spin', 
            'OS_usage', 'OS_velo', 'OS_spin', 
            'fb_thrown', 'bb_thrown', 'os_thrown'
            ]

best_models = []
train_df, test_df = train_test_split(df_balanced, stratify=df_balanced['is_workload_inj'], test_size=0.2, random_state=42)

# --- 2. Backward selection on training set ---
def backward_elimination(data, target, features, significance_level=0.05):
    remaining = features.copy()
    while len(remaining) > 0:
        formula = f"{target} ~ {' + '.join(remaining)}"
        model = smf.logit(formula=formula, data=data).fit(disp=False)
        pvalues = model.pvalues.iloc[1:]  # exclude intercept
        max_pval = pvalues.max()
        if max_pval > significance_level:
            worst_feature = pvalues.idxmax()
            remaining.remove(worst_feature)
        else:
            break
    final_model = smf.logit(f"{target} ~ {' + '.join(remaining)}", data=data).fit(disp=False)
    return final_model, remaining

best_model, kept_features = backward_elimination(train_df, target, features)

print("Final features kept:", kept_features)
print(best_model.summary())

# --- 3. Evaluate on test set ---
test_df = test_df.copy()
test_df["pred_prob"] = best_model.predict(test_df)
test_df["predicted"] = (test_df["pred_prob"] > 0.5).astype(int)

accuracy = (test_df["predicted"] == test_df[target]).mean()
print(f"Test accuracy: {accuracy:.3f}")

In [ ]:
cm = confusion_matrix(test_df[target], test_df["predicted"])
labels = ["No Injury", "Injury"]

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

# Optional: labeled display
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

# --- 5. ROC-AUC Score ---
roc_auc = roc_auc_score(test_df[target], test_df["pred_prob"])
print(f"ROC-AUC Score: {roc_auc:.3f}")

# --- 6. ROC Curve Plot ---
fpr, tpr, thresholds = roc_curve(test_df[target], test_df["pred_prob"])
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Logistic Model (Backward Selection)')
plt.legend(loc='lower right')
plt.show()

In [ ]:
final_features = ['days_since_last_start', 'pitches_last_start', 'number_start', 'FB_velo', 'fb_thrown']

# --- 1. Split data ---
train_df, test_df = train_test_split(df_balanced, stratify=df_balanced['is_workload_inj'], 
                                     test_size=0.2, random_state=42)

# --- 2. Standardize only predictor columns ---
scaler = StandardScaler()

train_scaled = train_df.copy()
test_scaled = test_df.copy()

train_scaled[final_features] = scaler.fit_transform(train_df[final_features])
test_scaled[final_features] = scaler.transform(test_df[final_features])

# --- 3. Train same model on standardized training data ---
formula = f"{target} ~ {' + '.join(final_features)}"
best_model_std = smf.logit(formula=formula, data=train_scaled).fit()

print(best_model_std.summary())

# --- 4. Evaluate on standardized test data ---
test_scaled["pred_prob"] = best_model_std.predict(test_scaled)
test_scaled["predicted"] = (test_scaled["pred_prob"] > 0.5).astype(int)

accuracy = (test_scaled["predicted"] == test_scaled[target]).mean()
print(f"Test accuracy (standardized model): {accuracy:.3f}")

In [ ]:
cm = confusion_matrix(test_scaled[target], test_scaled["predicted"])
labels = ["No Injury", "Injury"]

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Standardized Model)")
plt.show()

# --- 6. ROC-AUC ---
roc_auc = roc_auc_score(test_scaled[target], test_scaled["pred_prob"])
print(f"ROC-AUC Score (standardized): {roc_auc:.3f}")

# --- 7. ROC Curve ---
fpr, tpr, thresholds = roc_curve(test_scaled[target], test_scaled["pred_prob"])
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0,1],[0,1],'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Standardized Model)')
plt.legend(loc='lower right')
plt.show()